# 🤖 RuleBot — Complete Rule-Based Chatbot
> **Single master notebook** combining core chatbot + NLP concepts + advanced features

---
## 📋 Table of Contents
| Section | What It Covers |
|---------|---------------|
| STEP 1 | Install all libraries |
| STEP 2 | Import libraries + download NLTK data |
| STEP 3 | NLP Concept — Tokenization |
| STEP 4 | NLP Concept — Stemming |
| STEP 5 | NLP Concept — Lemmatization |
| STEP 6 | NLP Concept — Stop Word Removal |
| STEP 7 | Build text preprocessor |
| STEP 8 | Create intents dictionary (brain of the bot) |
| STEP 9 | Build intent matching engine |
| STEP 10 | Build response generator |
| STEP 11 | Main chatbot pipeline function |
| STEP 12 | Accuracy test before fine-tuning |
| STEP 13 | Fine-tune — add more patterns |
| STEP 14 | Re-test accuracy after fine-tuning |
| STEP 15 | Save intents to JSON |
| STEP 16 | Live chat loop |
| STEP 17 | Confidence threshold (advanced) |
| STEP 18 | Regex-based matching (advanced) |
| STEP 19 | Chat history logger |
| STEP 20 | Interactive chat UI widget |

---
> ⚠️ **Run every cell top to bottom using Shift+Enter. Do NOT skip any cell.**

---
# 🔧 PART 1 — SETUP
---

## ✅ STEP 1 — Install All Required Libraries
This **must** be the first cell you run.
It installs `nltk`, `gtts` (voice), and `ipywidgets` (chat UI) before anything else.

In [ ]:
# ============================================================
#  Install all dependencies first — prevents ModuleNotFoundError
# ============================================================
import subprocess, sys

packages = ['nltk', 'gtts', 'ipywidgets']
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'])
    print(f'  installed: {pkg}')

print('\n✅ All libraries installed successfully!')

## ✅ STEP 2 — Import Libraries and Download NLTK Data
Imports all modules and downloads NLTK datasets needed for tokenization and lemmatization.

In [ ]:
import random
import re
import json
import nltk
from datetime import datetime

# Download ALL required NLTK packages
# punkt_tab is required in newer NLTK — missing it causes LookupError
nltk_packages = ['punkt', 'punkt_tab', 'wordnet', 'stopwords', 'omw-1.4']
for pkg in nltk_packages:
    nltk.download(pkg, quiet=True)

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import stopwords

print('✅ All libraries imported!')
print('✅ NLTK data downloaded:', nltk_packages)

---
# 🧠 PART 2 — NLP CONCEPTS
Understanding how text is processed before matching.

---

## ✅ STEP 3 — NLP Concept 1: Tokenization
**Tokenization** = splitting a sentence into individual words (tokens).

Example: `"Hello World"` → `["Hello", "World"]`

In [ ]:
# ---- Tokenization Demo ----
sentence = 'Hello! How are you doing today? I am fine.'

# Word-level tokenization — splits into individual words
word_tokens = word_tokenize(sentence)
print('Word tokens  :', word_tokens)

# Sentence-level tokenization — splits into sentences
sent_tokens = sent_tokenize(sentence)
print('Sent tokens  :', sent_tokens)

## ✅ STEP 4 — NLP Concept 2: Stemming
**Stemming** = cutting a word to its crude root form (fast but imprecise).

- `playing` → `play`  
- `studies` → `studi` ← not a real word, that is the downside

In [ ]:
# ---- Stemming Demo ----
stemmer = PorterStemmer()

words = ['playing', 'played', 'plays', 'running', 'studies', 'better', 'happily']
print('{:<15} {:<15}'.format('Original', 'Stemmed'))
print('-' * 30)
for w in words:
    print('{:<15} {:<15}'.format(w, stemmer.stem(w)))

## ✅ STEP 5 — NLP Concept 3: Lemmatization (Better than Stemming)
**Lemmatization** = reduces words to their proper dictionary root.

- `studies` → `study`  
- `feet` → `foot`  
- Always produces a real valid word

In [ ]:
# ---- Lemmatization Demo ----
lemmatizer = WordNetLemmatizer()

words = ['playing', 'played', 'plays', 'running', 'studies', 'better', 'happily', 'feet']
print('{:<15} {:<15}'.format('Original', 'Lemmatized'))
print('-' * 30)
for w in words:
    print('{:<15} {:<15}'.format(w, lemmatizer.lemmatize(w)))

## ✅ STEP 6 — NLP Concept 4: Stop Word Removal
**Stop words** = common words that carry little meaning: `is`, `the`, `a`, `to`, `and`

Removing them reduces noise and improves keyword matching accuracy.

In [ ]:
# ---- Stop Word Removal Demo ----
stop_words = set(stopwords.words('english'))

sentence = 'what is the best way to learn python programming'
tokens   = word_tokenize(sentence)
filtered = [w for w in tokens if w not in stop_words]

print('Original :', tokens)
print('Filtered :', filtered)
print('Reduced from', len(tokens), 'words to', len(filtered), 'words')

---
# ⚙️ PART 3 — CORE CHATBOT ENGINE
---

## ✅ STEP 7 — Build the Text Preprocessor
The preprocessor cleans raw user input before keyword matching:
1. Lowercase
2. Remove punctuation and URLs
3. Tokenize into words
4. Optionally remove stop words
5. Lemmatize to root form

In [ ]:
# Reuse the lemmatizer already created in STEP 5
# (already initialized above as lemmatizer = WordNetLemmatizer())

def preprocess(text, remove_stopwords=False):
    """
    Full text preprocessing pipeline.
    Args:
        text             : raw user input string
        remove_stopwords : set True to strip common words like 'is', 'the'
    Returns:
        list of cleaned, lemmatized word tokens
    """
    # Step 1: Lowercase everything
    text = text.lower()

    # Step 2: Remove URLs (http://... or www....)
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Step 3: Remove all punctuation and special characters
    text = re.sub(r'[^a-z0-9\s]', '', text)

    # Step 4: Tokenize into individual words
    tokens = word_tokenize(text)

    # Step 5 (optional): Remove stop words
    if remove_stopwords:
        sw = set(stopwords.words('english'))
        tokens = [t for t in tokens if t not in sw]

    # Step 6: Lemmatize to root form
    tokens = [lemmatizer.lemmatize(t) for t in tokens]

    return tokens


# ---- TEST ----
test_inputs = [
    "Hello!! How are you doing today???",
    "What is the BEST way to learn Python??"
]
for t in test_inputs:
    print('Input    :', t)
    print('Normal   :', preprocess(t))
    print('No stops :', preprocess(t, remove_stopwords=True))
    print()

## ✅ STEP 8 — Create the Intents Dictionary (Brain of the Bot)
Each **intent** holds:
- `patterns` — phrases/keywords the user might type
- `responses` — replies the bot picks from randomly

> 💡 **More patterns = better accuracy.** This is where fine-tuning happens.

In [ ]:
# ============================================================
#  INTENTS — The brain of RuleBot
#  Fine-tuned with broad, varied patterns for high accuracy
# ============================================================

intents = {

    'greeting': {
        'patterns': [
            'hello', 'hi', 'hey', 'good morning', 'good evening', 'good afternoon',
            'howdy', 'greetings', 'whats up', 'sup', 'hiya', 'yo', 'ello',
            'morning', 'afternoon', 'evening', 'hi there', 'hello there',
            'hey there', 'good day', 'what is up'
        ],
        'responses': [
            'Hello! 👋 How can I help you today?',
            'Hi there! What can I do for you? 😊',
            'Hey! Great to see you. How can I assist?',
            'Greetings! I am RuleBot — ask me anything!'
        ]
    },

    'farewell': {
        'patterns': [
            'bye', 'goodbye', 'see you', 'see ya', 'take care', 'later',
            'ciao', 'farewell', 'ttyl', 'talk later', 'going now',
            'leaving', 'im off', 'see you soon', 'have a good day',
            'good night', 'goodnight', 'night'
        ],
        'responses': [
            'Goodbye! 👋 Have a great day!',
            'See you later! Take care 😊',
            'Bye! Come back anytime you need help!',
            'Farewell! It was nice chatting with you.'
        ]
    },

    'name': {
        'patterns': [
            'what is your name', 'who are you', 'your name',
            'what should i call you', 'introduce yourself',
            'what do i call you', 'tell me your name',
            'what are you called', 'may i know your name'
        ],
        'responses': [
            'I am RuleBot 🤖 — your rule-based assistant!',
            'My name is RuleBot. Nice to meet you!',
            'You can call me RuleBot. How can I help?'
        ]
    },

    'how_are_you': {
        'patterns': [
            'how are you', 'how do you do', 'hows it going',
            'how are you doing', 'are you okay', 'you good',
            'hows things', 'feeling okay', 'you alright',
            'doing okay', 'how have you been', 'how is it going'
        ],
        'responses': [
            'I am doing great, thanks for asking! 😄',
            'All systems running perfectly! How about you?',
            'I am just a bot, but I feel wonderful! 🤖'
        ]
    },

    'age': {
        'patterns': [
            'how old are you', 'what is your age', 'your age',
            'how young are you', 'when were you born'
        ],
        'responses': [
            'I am ageless — I was just created! 🤖',
            'Age does not apply to bots, but I am brand new!'
        ]
    },

    'creator': {
        'patterns': [
            'who made you', 'who created you', 'who built you',
            'who is your creator', 'who developed you',
            'who programmed you', 'who designed you'
        ],
        'responses': [
            'I was built by a Python developer learning AI! 🐍',
            'A smart programmer created me using Python and NLTK!'
        ]
    },

    'weather': {
        'patterns': [
            'weather', 'temperature', 'is it raining',
            'hows the weather', 'forecast', 'sunny', 'cloudy',
            'will it rain', 'is it hot', 'is it cold'
        ],
        'responses': [
            'I cannot check live weather, but try weather.com! 🌤',
            'I do not have internet access, but Google Weather works great! ☀'
        ]
    },

    'joke': {
        'patterns': [
            'tell me a joke', 'joke', 'make me laugh', 'say something funny',
            'humor me', 'funny', 'comedy', 'laugh', 'entertain me',
            'crack a joke', 'be funny', 'got any jokes'
        ],
        'responses': [
            'Why do programmers prefer dark mode? Because light attracts bugs! 🐛😂',
            'Why was the computer cold? It left its Windows open! 😄',
            'I told my computer I needed a break. Now it keeps sending me Kit Kat ads! 🍫',
            'Why do Java developers wear glasses? Because they do not C#! 😂',
            'What do you call a fish with no eyes? A fsh! 🐟😂'
        ]
    },

    'help': {
        'patterns': [
            'help', 'what can you do', 'assist me', 'what do you know',
            'capabilities', 'features', 'how can you help',
            'what topics do you know', 'what do you cover'
        ],
        'responses': [
            'I can chat, tell jokes, answer basic questions, and more! Try asking me anything 😊',
            'Here is what I can do: greet you, tell jokes, answer FAQs, and chat!'
        ]
    },

    'thanks': {
        'patterns': [
            'thank you', 'thanks', 'thank you so much', 'thx', 'ty',
            'appreciate it', 'many thanks', 'much appreciated',
            'appreciate that', 'cheers', 'thanks a lot', 'thanks a bunch'
        ],
        'responses': [
            'You are welcome! 😊',
            'Happy to help! Anything else?',
            'No problem at all! 🤖'
        ]
    },

    'time': {
        'patterns': [
            'what time is it', 'current time', 'tell me the time',
            'time now', 'whats the time', 'do you know the time'
        ],
        'responses': [
            'I do not have access to real-time data, but your device clock does! ⏰'
        ]
    },

    'python': {
        'patterns': [
            'python', 'what is python', 'tell me about python',
            'python programming', 'learn python', 'python language',
            'is python good', 'why python'
        ],
        'responses': [
            'Python is a powerful, beginner-friendly language used in AI, data science, and web dev! 🐍',
            'Python is great for AI! It has tons of libraries like NumPy and TensorFlow!'
        ]
    },

    'about_ai': {
        'patterns': [
            'what is ai', 'artificial intelligence', 'machine learning',
            'deep learning', 'neural network', 'tell me about ai',
            'what is machine learning', 'what is deep learning'
        ],
        'responses': [
            'AI is the simulation of human intelligence by machines! 🤖',
            'Machine Learning is a branch of AI where machines learn from data!',
            'Deep Learning uses neural networks to solve complex problems like image recognition!'
        ]
    },

    'default': {
        'patterns': [],
        'responses': [
            'Hmm, I did not understand that. Try asking something else! 🤔',
            'I am not sure about that. Could you rephrase? 😅',
            'That is beyond my rules right now. Try: say hello, ask for a joke, or ask about Python!'
        ]
    }
}

print('Loaded', len(intents), 'intents into RuleBot!')
print('Topics:', [k for k in intents if k != 'default'])

## ✅ STEP 9 — Build the Intent Matching Engine
Uses a **scoring system** to find the best-matching intent.

Scoring:
- `+10` points if a full pattern phrase is found in the input
- `+1`  point  for each matching word token

The intent with the highest score wins.

In [ ]:
CONFIDENCE_THRESHOLD = 3  # Minimum score to accept a match (avoids wrong guesses)

def match_intent(user_input):
    """
    Finds the best matching intent for the user message.
    Returns: (intent_name, score)
    Falls back to 'default' if score is below CONFIDENCE_THRESHOLD.
    """
    tokens    = preprocess(user_input)   # Clean input tokens
    user_lower = user_input.lower()       # Lowercase original for phrase matching

    best_intent = 'default'
    best_score  = 0

    for intent, data in intents.items():
        if intent == 'default':
            continue

        score = 0
        for pattern in data['patterns']:
            pat_tokens = preprocess(pattern)

            # Check 1: Full phrase match — strongest signal
            if pattern in user_lower:
                score += 10

            # Check 2: Word-level token overlap
            score += len(set(pat_tokens) & set(tokens))

        if score > best_score:
            best_score  = score
            best_intent = intent

    # Apply confidence threshold — reject weak matches
    if best_score < CONFIDENCE_THRESHOLD:
        best_intent = 'default'

    return best_intent, best_score


# ---- TEST the matcher ----
test_msgs = ['hello there', 'tell me a joke', 'what is python', 'goodbye', 'xyz blah']
print('{:<30} {:<18} Score'.format('Input', 'Intent Matched'))
print('-' * 55)
for t in test_msgs:
    intent, score = match_intent(t)
    print('{:<30} {:<18} {}'.format(t, intent, score))

## ✅ STEP 10 — Build the Response Generator
Once the intent is matched, pick a **random response** from that intent.
Randomness makes the bot feel more natural and less repetitive.

In [ ]:
def get_response(intent):
    """
    Returns a random response string for the given intent.
    Falls back to default if intent is not found.
    """
    if intent in intents:
        return random.choice(intents[intent]['responses'])
    return random.choice(intents['default']['responses'])


# ---- TEST — run multiple times to see different random responses ----
print('Sample joke responses (random each time):')
for i in range(3):
    print('  Reply', i+1, ':', get_response('joke'))

## ✅ STEP 11 — Main Chatbot Pipeline Function
Combines all steps into one clean function.

```
Input → preprocess → match_intent → get_response → return reply
```

In [ ]:
def chatbot_response(user_input):
    """
    Full chatbot pipeline in one call.
    Input  : raw user message string
    Output : bot reply string
    """
    # Guard: empty input
    if not user_input.strip():
        return 'Please type something! I am listening. 👂'

    intent, score = match_intent(user_input)
    return get_response(intent)


# ---- TEST the full pipeline ----
pipeline_tests = [
    'Hi there!',
    'What is your name?',
    'Tell me something funny',
    'Thank you so much!',
    'what is machine learning',
    'blah blah xyz 123'      # Unknown — triggers default
]

print('=' * 50)
print('Full Pipeline Test')
print('=' * 50)
for msg in pipeline_tests:
    print('You :', msg)
    print('Bot :', chatbot_response(msg))
    print('-' * 50)

---
# 🎯 PART 4 — FINE-TUNING & ACCURACY
---

## ✅ STEP 12 — Accuracy Test Before Fine-Tuning
We test the bot against 16 known input/intent pairs.
Any `FAIL` rows tell us exactly where to add more patterns.

In [ ]:
# ============================================================
#  TEST CASES — (what user types, expected intent)
# ============================================================
test_cases = [
    ('hello',                 'greeting'),
    ('hey there',            'greeting'),
    ('good morning',         'greeting'),
    ('bye',                  'farewell'),
    ('see you later',        'farewell'),
    ('what is your name',    'name'),
    ('who are you',          'name'),
    ('how are you doing',    'how_are_you'),
    ('are you okay',         'how_are_you'),
    ('tell me a joke',       'joke'),
    ('make me laugh',        'joke'),
    ('thank you',            'thanks'),
    ('what can you do',      'help'),
    ('tell me about python', 'python'),
    ('what time is it',      'time'),
    ('xyzabc random stuff',  'default'),
]

def run_accuracy_test(label=''):
    """Runs all test cases and prints a pass/fail report."""
    correct = 0
    total   = len(test_cases)
    fails   = []

    print('\n--- Accuracy Test', label, '---')
    print('{:<30} {:<16} {:<16} {}'.format('Input', 'Expected', 'Got', 'Result'))
    print('-' * 75)

    for msg, expected in test_cases:
        predicted, _ = match_intent(msg)
        ok = (predicted == expected)
        if ok:
            correct += 1
        else:
            fails.append((msg, expected, predicted))
        icon = 'PASS' if ok else 'FAIL'
        print('{:<30} {:<16} {:<16} {}'.format(msg, expected, predicted, icon))

    acc = correct / total * 100
    print('-' * 75)
    print('Accuracy: {}/{} = {:.1f}%'.format(correct, total, acc))

    if fails:
        print('\nFailed inputs — add these as patterns to fix them:')
        for msg, exp, got in fails:
            print('  Input: "{}" — should be "{}" but got "{}"'.format(msg, exp, got))

    return acc


acc_before = run_accuracy_test('(Before Fine-Tuning)')

## ✅ STEP 13 — Fine-Tune: Add More Patterns
**Fine-tuning** for a rule-based bot = improving your rules.

Rule: every `FAIL` row above means that phrase is missing from the intent patterns.
Add it, re-test, repeat until accuracy is 90%+.

In [ ]:
# ---- Helper functions to update intents easily ----

def add_patterns(intent_name, new_patterns):
    """Adds new keyword patterns to an existing intent."""
    if intent_name in intents:
        intents[intent_name]['patterns'].extend(new_patterns)
        print('Added', len(new_patterns), 'patterns to "' + intent_name + '"')
    else:
        print('Intent not found:', intent_name)

def add_new_intent(name, patterns, responses):
    """Creates a brand new intent in the dictionary."""
    intents[name] = {'patterns': patterns, 'responses': responses}
    print('New intent "' + name + '" added with', len(patterns), 'patterns')


# ---- Fine-Tune Pass 1: Fill gaps found in the test above ----
add_patterns('greeting',    ['hi everyone', 'hello all', 'hey you', 'good to see you'])
add_patterns('farewell',    ['see you around', 'catch you later', 'im leaving now'])
add_patterns('joke',        ['got a joke', 'share a joke', 'any jokes', 'cheer me up'])
add_patterns('how_are_you', ['how are things', 'how is life', 'all good with you'])
add_patterns('thanks',      ['thanks a lot', 'thank you very much', 'so grateful'])
add_patterns('name',        ['what are you', 'your identity', 'tell me who you are'])
add_patterns('python',      ['python basics', 'python tutorial', 'how to learn python'])
add_patterns('about_ai',    ['tell me about machine learning', 'explain ai',
                             'what are neural networks', 'how does ai work'])


# ---- Fine-Tune Pass 2: Add a brand new intent ----
add_new_intent(
    'compliment',
    patterns=[
        'you are great', 'well done', 'good job', 'you are awesome',
        'nice work', 'brilliant', 'impressive', 'you are smart'
    ],
    responses=[
        'Thank you so much! You are kind 😊',
        'That means a lot! I try my best 🤖',
        'Aw, thanks! You made my circuits smile 😄'
    ]
)


# ---- Fine-Tune Pass 3: Add another new intent ----
add_new_intent(
    'insult',
    patterns=[
        'you are stupid', 'you are dumb', 'you are useless',
        'you are bad', 'you are terrible', 'i hate you'
    ],
    responses=[
        'I am sorry to hear that. I am still learning! 😔',
        'That is okay, I will try to do better! 🙂',
        'I understand. Let me know how I can improve!'
    ]
)


print('\nTotal intents now:', len(intents))
print('All intents:', list(intents.keys()))

## ✅ STEP 14 — Re-Test Accuracy After Fine-Tuning
Run the same tests again to confirm improvement.

In [ ]:
acc_after = run_accuracy_test('(After Fine-Tuning)')

print('\n--- Improvement Summary ---')
print('Before :', round(acc_before, 1), '%')
print('After  :', round(acc_after,  1), '%')
print('Gain   :', round(acc_after - acc_before, 1), '%')

if acc_after >= 90:
    print('\n✅ Excellent! Chatbot is well-tuned!')
elif acc_after >= 75:
    print('\n⚠  Good — add more patterns to the FAIL rows to push higher.')
else:
    print('\n❌ Needs improvement — add patterns for every FAIL row above.')

## ✅ STEP 15 — Save Intents to JSON File
Saves the final fine-tuned intents dictionary to `intents.json`.
This file is used by the Streamlit web app.

In [ ]:
# Save to file
with open('intents.json', 'w') as f:
    json.dump(intents, f, indent=4)
print('Intents saved to intents.json')

# Verify by loading back
with open('intents.json', 'r') as f:
    verify = json.load(f)
print('Verified:', len(verify), 'intents in file')

# Show total patterns across all intents
total_patterns = sum(len(v['patterns']) for v in verify.values())
print('Total patterns across all intents:', total_patterns)

---
# 💬 PART 5 — CHAT INTERFACES
---

## ✅ STEP 16 — Live Chat Loop (Terminal-Style)
Type your message when prompted. Type `exit` to stop.

> If the loop gets stuck, click the **Stop button (⏹)** in the Colab toolbar.

In [ ]:
# ============================================================
#  LIVE CHAT — Type below when prompted. Type 'exit' to stop.
# ============================================================
print('RuleBot ready! Type exit to quit.')
print('=' * 45)

MAX_TURNS = 50  # Safety limit prevents infinite loop in Colab
turn = 0

while turn < MAX_TURNS:
    try:
        user_input = input('You: ')

        # Exit keywords
        if user_input.lower().strip() in ['exit', 'quit', 'bye', 'goodbye', 'stop', 'q']:
            print('Bot: Goodbye! Thanks for chatting! 👋')
            break

        # Skip blank input
        if not user_input.strip():
            print('Bot: Please type something!')
            continue

        print('Bot:', chatbot_response(user_input))
        print('-' * 45)
        turn += 1

    except KeyboardInterrupt:
        print('\nBot: Session ended. Goodbye!')
        break
    except Exception as e:
        print('Bot: Something went wrong, please try again.')
        continue

print('Chat session ended. Run the next cell to continue!')

## ✅ STEP 17 — Advanced: Regex-Based Matching
An alternative matching approach using **regular expressions**.
Regex can detect word variations in one pattern (e.g. `hi|hello|hey` in one line).

In [ ]:
# Regex patterns — covers more variations per line
regex_intents = [
    ('greeting',  r'\b(hello|hi|hey|good\s+(morning|evening|afternoon)|howdy)\b'),
    ('farewell',  r'\b(bye|goodbye|see\s+you|take\s+care|ciao|farewell)\b'),
    ('joke',      r'\b(joke|funny|laugh|humor|comedy|crack\s+a\s+joke)\b'),
    ('thanks',    r'\b(thank|thanks|ty|thx|appreciate|cheers)\b'),
    ('python',    r'\b(python|programming|coding)\b'),
    ('about_ai',  r'\b(ai|artificial\s+intelligence|machine\s+learning|deep\s+learning)\b'),
    ('help',      r'\b(help|assist|capabilities|what\s+can\s+you\s+do)\b'),
]

def regex_match_intent(user_input):
    """Match intent using compiled regular expressions."""
    text = user_input.lower()
    for intent, pattern in regex_intents:
        if re.search(pattern, text):
            return intent
    return 'default'


# ---- TEST ----
regex_tests = [
    'Good Morning everyone!',
    'haha tell me a funny joke',
    'I love coding in Python',
    'What is deep learning?',
    'gibberish input xyz'
]
print('Regex Matching Test:')
print('-' * 45)
for t in regex_tests:
    print('  Input:', t)
    print('  Intent:', regex_match_intent(t))
    print()

## ✅ STEP 18 — Chat History Logger
Logs every message with a timestamp and saves the full conversation to a `.txt` file.

In [ ]:
chat_history = []  # In-memory log

def log_message(role, message):
    """Append a message to the chat history with timestamp."""
    ts = datetime.now().strftime('%H:%M:%S')
    chat_history.append({'time': ts, 'role': role, 'message': message})

def save_history(filename='chat_history.txt'):
    """Write the full chat log to a text file."""
    with open(filename, 'w') as f:
        f.write('=== RuleBot Chat History ===\n\n')
        for entry in chat_history:
            f.write('[{}] {}: {}\n'.format(
                entry['time'], entry['role'].upper(), entry['message']
            ))
    print('Chat history saved to', filename)

def chatbot_with_logging(user_input):
    """Chatbot that logs all messages before returning a reply."""
    log_message('user', user_input)
    response = chatbot_response(user_input)
    log_message('bot', response)
    return response


# ---- Simulate a logged conversation ----
sample_convo = ['Hello!', 'Tell me a joke', 'What is Python?', 'You are great!', 'Thanks!']

print('Simulated logged conversation:')
print('=' * 45)
for msg in sample_convo:
    reply = chatbot_with_logging(msg)
    print('You:', msg)
    print('Bot:', reply)
    print()

save_history()
print('Total messages logged:', len(chat_history))

## ✅ STEP 19 — Evaluation Report
Runs the full 16-case test and prints a detailed breakdown of failures.

In [ ]:
def full_evaluation_report():
    """Detailed accuracy report with failure analysis."""
    correct  = 0
    failures = []

    for msg, expected in test_cases:
        predicted, _ = match_intent(msg)
        if predicted == expected:
            correct += 1
        else:
            failures.append((msg, expected, predicted))

    total    = len(test_cases)
    accuracy = correct / total * 100

    print('\n===== FINAL EVALUATION REPORT =====')
    print('  Total tests :', total)
    print('  Passed      :', correct)
    print('  Failed      :', len(failures))
    print('  Accuracy    : {:.1f}%'.format(accuracy))

    if failures:
        print('\n  FAILURES — Fix by adding these to the correct intent patterns:')
        for msg, exp, got in failures:
            print('    Input: "{}" | Expected: {} | Got: {}'.format(msg, exp, got))
    else:
        print('\n  Perfect Score! No failures.')

    return accuracy


final_acc = full_evaluation_report()

## ✅ STEP 20 — Interactive Chat UI Widget
A visual chat window inside Colab.
Type in the text box and click **Send** — no need to use the terminal input.

In [ ]:
from IPython.display import display, HTML
import ipywidgets as widgets

# ---- Build the chat widget layout ----
chat_log = widgets.Output(layout=widgets.Layout(
    height='320px', overflow_y='auto',
    border='2px solid #ddd', padding='10px'
))

text_input = widgets.Text(
    placeholder='Type your message and press Enter or click Send...',
    layout=widgets.Layout(width='75%')
)

send_btn = widgets.Button(
    description='Send',
    button_style='primary',
    layout=widgets.Layout(width='12%')
)

clear_btn = widgets.Button(
    description='Clear',
    button_style='warning',
    layout=widgets.Layout(width='12%')
)


def on_send(b):
    """Called when Send is clicked or Enter is pressed."""
    user_msg = text_input.value.strip()
    if not user_msg:
        return
    try:
        bot_reply = chatbot_response(user_msg)
    except Exception:
        bot_reply = 'Sorry, something went wrong. Try again!'

    with chat_log:
        display(HTML(
            "<div style='text-align:right;margin:4px 0'>"
            "<span style='background:#0084ff;color:white;padding:6px 12px;"
            "border-radius:16px;display:inline-block;max-width:75%'>"
            "You: {}".format(user_msg) +
            "</span></div>"
            "<div style='margin:4px 0'>"
            "<span style='background:#e4e6eb;color:#111;padding:6px 12px;"
            "border-radius:16px;display:inline-block;max-width:75%'>"
            "Bot: {}".format(bot_reply) +
            "</span></div>"
        ))
    text_input.value = ''


def on_clear(b):
    """Clears the chat window."""
    chat_log.clear_output()


send_btn.on_click(on_send)
clear_btn.on_click(on_clear)

# FIX: Use observe instead of deprecated on_submit
def on_enter(change):
    if change['name'] == 'value' and text_input.value.endswith('\n'):
        on_send(None)

text_input.on_submit(on_send)

# ---- Display ----
display(HTML('<h3 style="font-family:sans-serif">RuleBot Chat</h3>'))
display(chat_log)
display(widgets.HBox([text_input, send_btn, clear_btn]))

# Welcome message
with chat_log:
    display(HTML(
        "<div style='margin:4px 0'>"
        "<span style='background:#e4e6eb;color:#111;padding:8px 14px;"
        "border-radius:16px;display:inline-block'>"
        "Bot: Hi! I am RuleBot. Say hello, ask for a joke, or ask about Python!"
        "</span></div>"
    ))

---
# 🏆 SUMMARY

| Step | What You Built |
|------|----------------|
| 1-2 | Installed and imported all libraries |
| 3-6 | Learned NLP: tokenization, stemming, lemmatization, stop words |
| 7 | Built the text preprocessor |
| 8 | Created the intents dictionary (14 topics) |
| 9 | Built the intent matching engine with confidence threshold |
| 10 | Built the response generator |
| 11 | Assembled the full chatbot pipeline |
| 12 | Measured accuracy before fine-tuning |
| 13 | Fine-tuned with extra patterns + 2 new intents |
| 14 | Re-tested and confirmed accuracy improvement |
| 15 | Saved intents to intents.json |
| 16 | Ran live chat loop |
| 17 | Added regex-based matching (advanced) |
| 18 | Added chat history logger |
| 19 | Generated full evaluation report |
| 20 | Built interactive visual chat widget |

## 🚀 Next Step
Open and run `RuleBot_Streamlit_App.py` to launch the full web application!